**Claudi Vall Müller - University of Barcelona**


Last modified: 08/11/2024


## Functions

In [2]:
from scipy.integrate import quad
import numpy as np
from math import sin, cos, tan, radians, asin, acos, atan, pi
from astropy.io import fits
import pandas as pd
import matplotlib.pyplot as plt

# Numerical Constants
c = 299792458/1000  # km/s
H_0 = 70  # km/s / Mpc
O_m = 0.3


### Angular diameter distance calculator
def integrand(z, H_0, O_m):
    '''Integrand for the angular diameter distance integral.'''
    Hz = H_0 * (O_m * (1+z)**3 + (1-O_m))**0.5
    f = 1/Hz    
    return f

def D_A(z_1, z_2):
    '''Computes angular diameter distance from two different redshift points, z_2 > z_1. For example,
    z_1 = 0 (observer), z_2 = z_S for computing D_S.
    c is the speed of light in km/s, H0 is the hubble constant in km/s/Mpc and O_m is omega matter from the
    cosmological model. Angular diameter distance returned is in Mpc.'''
    
    integral = quad(integrand, z_1, z_2, args = (H_0, O_m))
    D = c/(1 + z_2) * np.array(integral)
    
    return D

### Magnification parameters rescaling

def redshift_rescale(k_tilda, z_L, z_S):    
    '''Rescales lens model value of kappa (or any other magnification parameter) (k_tilda) for a certain lens redshift (z_L) and source redshift (z_S).
        Returns rescaled value of kappa'''
    
    D_LS = D_A(z_L, z_S)
    D_S = D_A(0, z_S)

    kappa = k_tilda * D_LS[0]  / D_S[0]
    return kappa


def redshift_descale(kappa, z_L, z_S):
    '''Descales lens model from a certain redshift to D_LS/D_S = 1.'''
    D_LS = D_A(z_L, z_S)
    D_S = D_A(0, z_S)

    kappa_tilda = kappa / (D_LS[0]/D_S[0])
    return kappa_tilda

##  COORDINATES TRANSFORMATION
def RA_to_degrees(coordinates):
    '''Argument coordinates = [h, m, s] (right ascension coordinades). Returns degrees.'''
    h = coordinates[0]
    m = coordinates[1]
    s = coordinates[2]
    
    if abs(h) != 0:
        sign = h/abs(h)
    else:
        sign = m/abs(m)
    
    hours = abs(h) + m/60 + s/3600
    degrees = sign * hours * 15
    return degrees 

def DEC_to_degrees(coordinates):
    '''Agument coordinates = [deg, m, s] (sexagesimal coordinates). Returns degrees.'''
    deg = coordinates[0]
    m = coordinates[1]
    s = coordinates[2]
    
    if abs(deg) != 0:
        sign = deg/abs(deg)
    else:
        sign = m/abs(m)
        
    degrees = sign * (abs(deg) + m/60 + s/3600)
    return degrees

def degrees_to_RA(degrees):
    '''Converts degrees to RA coordinates (h, m, s).'''
    sign = degrees/abs(degrees)
    h = int(abs(degrees)/15) * sign
    m = int((abs(degrees)/15 - abs(h)) * 60)
    s = ((abs(degrees)/15 - abs(h))* 60 - m) * 60
    return h, m, s 

def degrees_to_DEC(degrees):
    '''Converts degrees to DEC sexagesimal coordinates (deg, m , s).'''
    sign = degrees/abs(degrees)
    deg = int(abs(degrees)) * sign
    m = int((abs(degrees) - abs(deg))*60)
    s = ((abs(degrees) - abs(deg))*60 - m)*60
    return deg, m, s
    
## Distance between two RA, DEC points

def distance(P1, P2):
    '''Computes distance between P1 = [RA1, DEC1] and P2 = [RA2, DEC2] in the small angle approximation. RA1 = [h, min, s].
    Example (Icarus position): P1 = [ [11, 49, 35.66], [22, 23, 48.00] ]. Distance is given in arcsec (").'''
    
    RA1, DEC1 = P1
    RA2, DEC2 = P2
    
    RA1_deg = RA_to_degrees(RA1)
    DEC1_deg = DEC_to_degrees(DEC1)
    RA2_deg = RA_to_degrees(RA2)
    DEC2_deg = DEC_to_degrees(DEC2)
    
    d = ( (RA1_deg - RA2_deg)**2 * (cos(radians(DEC1_deg)))**2  +  (DEC1_deg - DEC2_deg)**2  )**0.5
    d_arcsec = d * 3600
    
    return d_arcsec


# Pixel slope, pixel distance and analytic slope given magnification parameters
def pixel_distance(x1, y1, x2, y2, delta_x, delta_y):
    '''Distance between two points given the (x1, y1) and (x2, y2) pixel coordinates of the points
    and the x and y size of each pixel. Distance returned in the same units as the argument pixel size.'''
    return ((x1-x2)**2 * delta_x**2 + (y1-y2)**2 * delta_y**2)**0.5

def pixel_slope(x1, y1, x2, y2):
    '''Slope given two points (x1, y1) and (x2, y2)'''
    return (y2-y1)/(x2-x1)

def analitic_slope(lambda_v, gamma_v, eta_v):
    '''Slope of the line defined by the eigenvector of eigenvalue 1-κ-γ.
    Arguments are the needed magnification parameters.'''
    return (-lambda_v + gamma_v)/eta_v


##  MAGNIFICATION PARAMETERS CALCULATION
def position_to_pixel(source_position, center_position_deg, c_x, c_y, delta_RA, delta_DEC):
    '''Converts source position = [RA, DEC] to pixel (x,y) value in our data array.
    RA = [h, m, s] and DEC = [º, ', ''] is the source position and center_position_deg = [RA(deg), DEC(deg)] are 
    the coordinates of the pixel (c_x, c_y). delta_RA, delta_DEC are the sizes of each pixel
    in the horizontal and vertical directions (degrees) and delta_RA must NOT include the cosδ correction. 
    (delta_RA = CDELT1/cosδ < 0).'''
    
    # Source position degrees
    RA_deg = RA_to_degrees(source_position[0])
    DEC_deg = DEC_to_degrees(source_position[1])
    
    # Center_position
    RA_0_deg = center_position_deg[0]
    DEC_0_deg = center_position_deg[1]

    
    # Pixel position of the source with origin at the bottom left corner
    
    pixel_x = int(round( (RA_deg - RA_0_deg)/delta_RA + c_x - 1, 0) )
    pixel_y = int(round( (DEC_deg - DEC_0_deg)/delta_DEC + c_y - 1, 0 ))
    
    return pixel_x, pixel_y
    
    
def parameter_with_error(data_map, pix_tol_x, pix_tol_y, x, y, z_L, z_S):
    '''Computes the parameter at (x,y) position of the data_map. Error is given by the number of pixels that have
    the same RA, DEC coordinates, and should therefore be explored (pix_tol_x), (pix_tol_y). z_L and z_S are lens
    and source redshifts.'''
    # pix_tol_x is the number of horizontal pixels with the same RA, DEC coordinates. 
    # If resolution is very high, there are many pixels with the same RA, DEC coordinates, so all of them should be explored
    parameter_values = np.array([])
    for i in range(2*(pix_tol_x-1) + 1):   # we move tolerance pixels horizontally. If we had 3 pixels with same coords, we would move 2 each direction and we'd have computed 2*2+1 = 5 values.
        for j in range(2*(pix_tol_y-1) + 1): # and tolerance pixels vertically
            x_coord = x - (pix_tol_x-1) + i
            y_coord = y - (pix_tol_y-1) + j
            try:
                parameter_values = np.append( parameter_values, redshift_rescale(data_map[y_coord][x_coord], z_L, z_S) )
            except:
                print("The position x, y = ({}, {}) is out of our image.".format(y_coord, x_coord))
    parameter_source = redshift_rescale(data_map[y][x], z_L, z_S)
    parameter_error = max(abs(parameter_values - parameter_source))
    return parameter_source, parameter_error
    
def magnification_parameters(kappa_map, gamma_map, lambda_map, eta_map, z_L, z_S, source_position, center_position_deg,
                             c_x, c_y, delta_RA, delta_DEC, shear_info):
    '''Computes magnification parameters (kappa, gamma, lambda, eta) at the source position = [RA, DEC] at corresponding
    lens and source redshifts (z_L, z_S).
    Arguments: data maps for each parameter, center_position_deg = [RA, DEC] (degrees) is the position of the 
    reference point, which we know corresponds to pixel position (c_x, c_y). delta_RA and delta_DEC are the dimensions
    of each pixel in degrees. delta_RA = delta_x/cosδ < 0. shear_info = True if the opened model provides shear parameters.'''

    
    x, y = position_to_pixel(source_position, center_position_deg, c_x, c_y, delta_RA, delta_DEC)

    
    tol = 0.01  # arcsec error.   SET tol = 0 if no error is wanted to be considered
     
    # Number of pixels that we can on average move and we will still be on our source position
    pix_tol_y = abs(int(round( (tol/3600)/delta_DEC, 0 )))  # how many pixels have same RA, DEC?
    
    if pix_tol_y < 2:   
        pix_tol_y = 2  # we want to move at least 1 pixel to each side (as if 3 pixels had same coordinates)
                        # we move now pix_tol_y - 1 pixels left and same number right for being sure we explore all pixels
                        # with the same coordinates
        
    pix_tol_x = abs(int(round( (15*tol/3600)/delta_RA, 0 )))
    if pix_tol_x < 2:
        pix_tol_x = 2  
    
    # Parameters computation with errors
        
    kappa_value, kappa_err   = parameter_with_error(kappa_map, pix_tol_x, pix_tol_y, x, y, z_L, z_S)
    gamma_value, gamma_err   = parameter_with_error(gamma_map, pix_tol_x, pix_tol_y, x, y, z_L, z_S)
    kappa_v = [kappa_value, kappa_err]
    gamma_v = [gamma_value, gamma_err]

    if shear_info == True:
        lambda_value, lambda_err = parameter_with_error(lambda_map, pix_tol_x, pix_tol_y, x, y, z_L, z_S)
        eta_value, eta_err       = parameter_with_error(eta_map,    pix_tol_x, pix_tol_y, x, y, z_L, z_S)
        lambda_v = [lambda_value, lambda_err]
        eta_v = [eta_value, eta_err]
    
    else:
        lambda_v = [0, 0]
        eta_v = [0, 0]
        
    return kappa_v, gamma_v, lambda_v, eta_v 
  
    
def mu_comput(kappa, gamma):
    '''Computes magnification and its associated error from kappa = [kappa_value, kappa_err]
    and gamma = [gamma_value, gamma_err] (convergence and shear values and errors).'''
    
    kappa_value, kappa_err = kappa
    gamma_value, gamma_err = gamma
    mu_value = 1/( (1-kappa_value)**2 - gamma_value**2  )
    mu_err = 2*(mu_value**2) * (   (1-kappa_value)**2 * kappa_err**2 + gamma_value**2 * gamma_err**2 )**0.5

    return mu_value, mu_err

    
def magnification_at_position(source_position, z_L, z_S, center_position_deg, c_x, c_y, delta_RA, delta_DEC, 
                              kappa_map, gamma_map, lambda_map, eta_map, shear_info):
    '''Computes magnification at source_position = [RA, DEC] with RA = [h, min, s], DEC = [º, ', "]
    at redshift z_L, z_S. ... '''
    
    kappa_v, gamma_v, lambda_v, eta_v = magnification_parameters(kappa_map, gamma_map, lambda_map, eta_map, 
                            z_L, z_S, source_position, center_position_deg, c_x, c_y, delta_RA, delta_DEC, shear_info)
    
    
    mu_value, mu_err = mu_comput(kappa_v, gamma_v)
    
    return mu_value, mu_err
    
    
    
    
### CRITICAL CURVE DETERMINATION

def critical_curve_loop(maps, slope, starting_position, theta_max, delta_theta, z_L, z_S, center_position, 
                        c_x, c_y, delta_RA, delta_DEC, shear_info, slope_number):
    
    '''Finds and returns the position (RA_c, DEC_c) of the nearest critical curve from some starting position along the
    direction of the eigenvector of eigenvalue 1-κ-γ, and the angular distance to this critical curve point (crit_dist). 
    Arguments are:
    maps = [kappa_map, gamma_map, lambda_map, eta_map] are the sky maps for the magnification parameters
    slope is the slope of the line of direction of corresponding eigenvalue
    starting_position = [RA_s, DEC_s]
    theta_max (") is the maximum angular distance from the starting position we want to study     
    delta_theta (") is the angular distance spacing between studied points until critical curve is reached
    z_L, z_S are lens/source redshifts
    center_position = [RA, DEC] is a reference position of the image with pixel position (c_x, c_y)
    delta_RA and delta_DEC are the dimensions of the pixels of the image
    shear_info = True if the model has λ and η information (second shear components).
    slope_number = 0 for lower bound (slope - error), 1 for central bound (slope) and 2 for upper bound (slope + error)
    '''
    kappa_map = maps[0]
    gamma_map = maps[1]
    lambda_map = maps[2]
    eta_map = maps[3]
    
    theta_max_deg = theta_max/3600   # arcsec to degrees
    delta_theta_deg = delta_theta/3600  # arcsec to degrees
    
    RA_s, DEC_s = starting_position    # starting position
    RA_s_deg, DEC_s_deg = [RA_to_degrees(RA_s), DEC_to_degrees(DEC_s)]
    
    
    # Maximum RA range to explore
    RA_1_deg = RA_s_deg - theta_max_deg / ( abs(cos(radians(DEC_s_deg))) *  ( 1 +  slope**2  )**0.5)
    RA_2_deg = RA_s_deg + theta_max_deg / ( abs(cos(radians(DEC_s_deg))) *  ( 1 +  slope**2  )**0.5)
    # obtained imposing distance θ and point on line    y-ys = -slope * (x-xs)
    
    delta_alpha_deg = delta_theta_deg / ( abs(cos(radians(DEC_s_deg))) *  ( 1 +  slope**2  )**0.5)
    # Δα obtained from Δθ projection to RA axis
    
    
    # Number of studied points
    N = int(round (2 * theta_max_deg/delta_theta_deg + 1 , 0))
    
    
    # dictionary where the computed magnifications are saved 
    magnifications_dict = {}
    
    # We don't want all of them to be printed.
    n_print = 50
    delta_print = (N-1)/(n_print - 1)   # pixel spacing between consecutive prints
    
    for i in range(0, N):
        RA_deg = RA_1_deg + i * delta_alpha_deg   
        RA = degrees_to_RA(RA_deg)
        DEC_deg = DEC_s_deg - slope * cos(radians(DEC_s_deg)) * (RA_deg - RA_s_deg)   # cos correction if eingenvector
        DEC = degrees_to_DEC(DEC_deg)                                                 # is given in (x, y) basis (not RA, DEC)
        study_position = [RA, DEC]                                                    # -slope sign since Δx = -Δα cos δ
        
        ### check study position
        dist = distance(starting_position, study_position)
        
        mag_i, mag_err = magnification_at_position(study_position, z_L, z_S, center_position, c_x, c_y, delta_RA, delta_DEC,
                  kappa_map, gamma_map, lambda_map, eta_map, shear_info)
        
        
        magnifications_dict[i] = mag_i
        
    # Study all computed magnifications for locating the sign change
    for k in range(len(magnifications_dict)):
        if k < (len(magnifications_dict) - 1):   
            if magnifications_dict[k] * magnifications_dict[k + 1] < 0: 
                if abs(magnifications_dict[k+1]) > abs(magnifications_dict[k]):  # we choose the maximum value
                    k = k+1
                # Caustic crossing coordinates
                RA_deg = RA_1_deg + k * delta_alpha_deg
                RA = degrees_to_RA(RA_deg)
                DEC_deg = DEC_s_deg - slope * (RA_deg - RA_s_deg) * cos(radians(DEC_s_deg)) 
                DEC = degrees_to_DEC(DEC_deg)
                crit_position = [RA, DEC] 

                crit_dist = distance(starting_position, crit_position)
                break
        
        if k == len(magnifications_dict) - 1:
            crit_position = 'Na'
            crit_dist = 'Na' 
    
    
    return crit_position
   
    
def critical_curve(maps, source_position, theta_max_0, delta_theta_0, tolerance, slope_0, z_L, z_S, center_position, c_x, c_y, 
                        delta_RA, delta_DEC, shear_info):
    '''Finds and returns the position (RA_c, DEC_c) of the nearest critical curve from source position along the
    direction of the eigenvector of eigenvalue 1-κ-γ, and the angular distance from source to critical curve (crit_dist). 
    Arguments are:
    maps = [kappa_map, gamma_map, lambda_map, eta_map]                  
    starting_position = [RA_s, DEC_s]
    theta_max_0 (") is the starting maximum angular distance from the source position we want to study     
    delta_theta_0 (") is the starting angular distance spacing between studied points until critical curve is reached
    tolerance is the precision at which we want the determination of the critical curve point
    z_L, z_S are lens/source redshifts
    center_position = [RA, DEC] is a reference position of the image with pixel position (c_x, c_y)
    delta_RA and delta_DEC are the dimensions of the pixels of the image
    shear_info = True if the model has λ and η information (second shear components).
    '''
    N_loops = 20     # some loops number that should never be reached (if no crossing is detected, we increase θ.
                                                      # if crossing is detected, we decrease Δθ until < tolerance)
    kappa_map = maps[0]
    gamma_map = maps[1]
    lambda_map = maps[2]
    eta_map = maps[3]
    
    # Source magnification parameters
    
    kappa_s, gamma_s, lambda_s, eta_s = magnification_parameters(kappa_map, gamma_map, lambda_map, eta_map,
                            z_L, z_S, source_position, center_position, c_x, c_y, delta_RA, delta_DEC, shear_info)    
    
    delta_slope = 0.001
    final_crit_positions = []   # final_crit_position_inf, final_crit_position, final_crit_position_sup
    final_crit_distances = []
    
    # Loop for finding upper and lower bound (considering error in slope) of critical curve position
    for k in range(0, 3):
        d_slope = (k-1) * delta_slope
        slope = slope_0 + d_slope

        # Starting values for the loop
        starting_position = source_position
        theta_max = theta_max_0
        delta_theta = delta_theta_0
        
        # When critical curve is reached, loop is repeated with lower spacing Δθ. Loop breaks when spacing reaches tolerance.
        for i in range(0, N_loops):
            
            crit_position = critical_curve_loop(maps, slope, starting_position, theta_max, delta_theta, z_L, z_S, 
                                                center_position, c_x, c_y, delta_RA, delta_DEC, shear_info, k)


            if delta_theta <= abs(tolerance):
                if crit_position == 'Na':
                    if k == 1:    # we only print info for the central value of the slope
                        print('No critical curve found.')    
                    break
            
                final_crit_position = crit_position
                final_crit_positions.append(crit_position)
                final_crit_dist = distance(final_crit_position, source_position)
                final_crit_distances.append(final_crit_dist)

                break  # end loop for a given slope
            
            
            if crit_position == 'Na':
                theta_max = theta_max*1.5
                continue
        
            # if Δθ can be reduced and we found critical curve crossing in the interval, repeat loop with lower θ and Δθ.
            starting_position = crit_position
            theta_max = 2*delta_theta   # new range to explore (easily seen that θ = 2*Δθ is a good option)
            delta_theta = delta_theta/10  # new resolution
            
            if  i == N_loops:
                if k == 1:    # we only print info for the central value of the slope
                    print('Not enough loops for finding critical curve with precision under given tolerance.')
                return 'Na', 'Na'
            
        
    return final_crit_positions, final_crit_distances


### d GRADIENT COMPUTATION

def d_grad_comput(maps, position_0, d_theta, slope_0, z_L, z_S, center_position_deg, c_x, c_y, delta_RA, delta_DEC, shear_info):
    '''Computes vec(d) = (-d(κ+λ)/dx1, -d(κ+λ)/dx2) on the basis of eigenvectors. x1 eigenvector of eigenvalue 1-κ-γ
    and x2 its perpendicular direction.
    Arguments are magnification parameters sky maps maps = [kappa_map, gamma_map, lambda_map, eta_map]
    position_0  = [RA, DEC] (RA = [h, min, sec], DEC = [º,',"]) is the position where we want to compute the d parameter
    d_theta (arcsec) is distance for computing derivatives
    z_L and z_S are the lens and source redshifts
    center_position_deg = [RA (deg), DEC(deg)] is a reference position of the sky map, with known pixel values (c_x, c_y)
    delta_RA and delta_DEC are the pixel width and height in degrees. delta_RA = delta_x/cosδ < 0
    shear_info = True if the selected model provides shear parameters (λ and η).
    Returns gradient d_grad = (d_grad_x1, d_grad_x2), gradient error d_grad_err = (d_grad_x1_err, d_grad_x2_err),
    and the modulus of the gradient with its error d = (d_mod, d_mod_err). All in arcsec^(-1).
    Slope of the line with direction of first eigenvector is (-λ+γ)/η and the slope of its perpendicular line is η/(λ-γ).
    '''
    
    kappa_map = maps[0]
    gamma_map = maps[1]
    lambda_map = maps[2]
    eta_map = maps[3]
    
    kappa_0, gamma_0, lambda_0, eta_0 = magnification_parameters(kappa_map, gamma_map, lambda_map, eta_map, 
                                                                 z_L, z_S, position_0, center_position_deg, 
                                                                 c_x, c_y, delta_RA, delta_DEC, shear_info)
    
    d_theta_deg = d_theta/3600  # arcsec to degrees 
    
    # Origin position
    RA_0, DEC_0 = position_0
    alpha_0 = RA_to_degrees(RA_0)
    delta_0 = DEC_to_degrees(DEC_0)
    
    # Slopes m- and m+ (fist and second eigenvectors)    
    slope_1 = slope_0    # x1 slope
    slope_2 = -1/(slope_0)   # perpendicular slope

    # We could include slope error by computing d in the three computed points. Here we only compute d in the central point.
     
    # Positive or negative direction of each eigenvector is unknown. For a smooth function, a change of sign here should
    # not affect the magnitude of the derivative, and therefore should not affect our final d value. 
    # The d vector will change its sign if we change the following sign.
    
    # Signs are + if eigenvectors (η, -λ+γ) and (λ-γ, η) point toward positive direction of x (→). This is not necessary,
    # since -(η, -λ+γ) is also a first eigenvector and has opposite direction. This choice of eigenvector is arbitrarely
#     sign_dx_1 = eta_0[0]/abs(eta_0[0])
#     sign_dx_2 = (lambda_0[0] - gamma_0[0])/abs(lambda_0[0] - gamma_0[0])
#     Change of sign since Δα = -Δx
#     sign_dalpha_1 = -sign_dx_1
#     sign_dalpha_2 = -sign_dx_2
    
    
    sign_dalpha_1 = 1     # a change of sign here should not affect the |d| value significantly
    sign_dalpha_2 = 1
    
    d_alpha_1 = sign_dalpha_1 * d_theta_deg / ( abs(cos(radians(delta_0))) * (1 + slope_1**2)**0.5 )
    d_alpha_2 = sign_dalpha_2 * d_theta_deg / ( abs(cos(radians(delta_0))) * (1 + slope_2**2)**0.5 )
    
    
    alpha_1 = alpha_0 + d_alpha_1
    delta_1 = delta_0 - slope_1 * cos(radians(delta_0)) * (alpha_1 - alpha_0)
    position_1 = [degrees_to_RA(alpha_1), degrees_to_DEC(delta_1)]
    
    kappa_1, gamma_1, lambda_1, eta_1 = magnification_parameters(kappa_map, gamma_map, lambda_map, eta_map, 
                                                                 z_L, z_S, position_1, center_position_deg, 
                                                                 c_x, c_y, delta_RA, delta_DEC, shear_info)  
    
    alpha_2 = alpha_0 + d_alpha_2
    delta_2 = delta_0 - slope_2 * cos(radians(delta_0)) * (alpha_2 - alpha_0)
    position_2 = [degrees_to_RA(alpha_2), degrees_to_DEC(delta_2)]
        
    x_0, y_0 = position_to_pixel(position_0, center_position_deg, c_x, c_y, delta_RA, delta_DEC)
    x_1, y_1 = position_to_pixel(position_1, center_position_deg, c_x, c_y, delta_RA, delta_DEC)
    x_2, y_2 = position_to_pixel(position_2, center_position_deg, c_x, c_y, delta_RA, delta_DEC)

    kappa_2, gamma_2, lambda_2, eta_2 = magnification_parameters(kappa_map, gamma_map, lambda_map, eta_map, 
                                                                 z_L, z_S, position_2, center_position_deg, 
                                                                 c_x, c_y, delta_RA, delta_DEC, shear_info)
    
    
    
    # Real distances between the studied positions
    # we see that in practice h_1 = h_2 = d_theta as expected.
    h_1 = distance(position_0, position_1)
    h_2 = distance(position_0, position_2)
        
    k_grad_x1 = (kappa_1[0] - kappa_0[0])/h_1 
    k_grad_x1_err = ((kappa_1[1]**2 + kappa_0[1]**2)**0.5)/h_1
    
    k_grad_x2 = (kappa_2[0] - kappa_0[0])/h_2 
    k_grad_x2_err = ((kappa_2[1]**2 + kappa_0[1]**2)**0.5)/h_2

    l_grad_x1 = (gamma_1[0] - gamma_0[0])/h_1
    l_grad_x1_err = ((gamma_1[1]**2 + gamma_0[1]**2)**0.5)/h_1

    l_grad_x2 = (gamma_2[0] - gamma_0[0])/h_2
    l_grad_x2_err = ((gamma_2[1]**2 + gamma_0[1]**2)**0.5)/h_2
 
    d_grad_x1 = -(k_grad_x1 + l_grad_x1)
    d_grad_x2 = -(k_grad_x2 + l_grad_x2)
    
    d_grad_x1_err = (k_grad_x1_err**2 + l_grad_x1_err**2)**0.5
    d_grad_x2_err = (k_grad_x2_err**2 + l_grad_x2_err**2)**0.5
    
    d_grad = (d_grad_x1, d_grad_x2)
    d_grad_err = (d_grad_x1_err, d_grad_x1_err)
    
    
    d_mod = (d_grad_x1**2 + d_grad_x2**2)**0.5
    d_mod_err = 1/d_mod * ( (d_grad_x1 * d_grad_x1_err)**2 + (d_grad_x2 * d_grad_x2_err)**2 )**0.5
    
    
    return d_grad, d_grad_err, (d_mod, d_mod_err)



# Kappa star computation
def kappa_star_comput(sigma_star, z_L, z_S):
    '''Computes k* = sigma*/sigma_crit. Arguments are sigma* (in M_sun/pc^2), redshift of the lens (z_l) and 
    redshift of the source (z_s). Returns k*.'''
    c = 299792458  # m/s
    G = 6.67430e-11 # N m^2 kg^-2
    
    M_sun = 1.98847e30 # kg 
    pc = 30856775814913673 # m
    
    D_S = D_A(0, z_S)
    D_L = D_A(0, z_L)
    D_LS = D_A(z_L, z_S)
    
    sigma_crit = c**2/(4*pi*G) * pc/M_sun * D_S/(D_L * D_LS) * 1e-6    # M_sun/pc^2
    kappa_star = sigma_star/sigma_crit
    
    return kappa_star




## Dictionaries

In [3]:
star_events = {1: 'LS1 (Icarus) lensed by MACSJ1149 (Kelly 2018)', 2: 'LS2 (NW Spock) lensed by MACSJ0416 (Rodney 2018)',
         3: 'LS3 (SE Spock) lensed by MACSJ0416 (Rodney 2018)', 4: 'LS4 (Warhol) lensed by MACSJ0416 (Kaurov 2019 / Chen 2019)',
         5: 'LS5 (Earendel) lensed by WHL0137 (Welch 2022)', 6: 'LS6 (Godzilla) lensed by PSZ1 G311.65 (Diego 2022)',
         7: 'LS7 (...) lensed by Abell 2744 (Chen 2022)', 8: 'LS8 (star-1) lensed by MACSJ0647 (Meena 2023)',
         9: 'LS9 (star-2) lensed by MACSJ0647 (Meena 2023)', 10: 'LS10 (Quyllur) lensed by El Gordo (Diego 2023)',
         11: 'LS11 (...) lensed by Abell 370 (Meena 2023)', 12: 'LS12 (Mothra) lensed by MACSJ0416 (Diego 2023)'}

clusters = {1: 'macs1149', 2: 'macs0416', 3: 'macs0416', 4: 'macs0416', 5: 'whl0137-08', 6: 'PSZ...', 7: 'abell2744', 
           8: 'macs0647', 9: 'macs0647', 10: 'ElGordo', 11: 'abell370', 12: 'macs0416'}

telescopes = {1: 'hlsp', 2: 'hlsp', 3: 'hlsp', 4: 'hlsp', 5: 'hlsp', 6: 'jwst', 7: 'hlsp', 8: 'hlsp', 
             9: 'hlsp', 10: 'jwst', 11: 'hlsp', 12: 'hlsp'}
programs = {1: 'frontier', 2: 'frontier', 3: 'frontier', 4: 'frontier', 5: 'relics', 6: '', 7: 'frontier', 8:'clash', 
           9: 'clash', 10: '', 11: 'frontier', 12: 'frontier'}
models = {1: 'bradac', 2: 'cats', 3: 'glafic', 4: 'merten', 5: 'sharon', 6: 'williams', 8: 'zitrin', 9: 'zitrin-ltm',
          10: 'zitrin-ltm-gauss', 11: 'zitrin-nfw', 12: 'diego', 13: 'caminha', 14: 'lenstool', 15: 'WSLAP',  16: 'keeton'}

versions = {1: '_v1', 2: '_v2', 2.1: '_v2.1', 3: '_v3', 4: '_v4', 4.1: '_v4.1', 4.2: '_v4cor'}


# Used models for each lensed case
model_ids_dict = {1: [2, 9, 3, 16], 2: [6, 1], 3: [6, 1], 4: [13, 3, 16, 5, 10, 11], 5: [9, 3, 15, 14], 7: [5],
                  8: [10, 11], 9: [10, 11], 10: [12], 11: [3, 6, 16], 12: [10]}
version_ids_dict = {1: [4.1, 1, 3, 4], 2: [4, 3], 3: [4,3], 4: [4, 4, 4, 4.2, 3, 3], 5: [1, 1, 1, 1], 7: [4.2],
                    8: [2, 2], 9: [2, 2], 10: [1], 11: [4, 4, 4], 12: [3]}


# Lens and source redshifts
lens_redshift = {1: 0.544, 2: 0.397, 3: 0.397, 4: 0.397, 5: 0.566, 6: 0.443, 7: 0.308, 8: 0.591, 9: 0.591, 10: 0.870, 11: 0.375, 12: 0.397}
source_redshift = {1: 1.49, 2: 1.0054, 3: 1.0054, 4: 0.9397, 5: 6.2, 6: 2.37, 7: 2.6501, 8: 4.8, 9: 4.8, 10: 2.1878, 11: 1.2567, 12: 2.091}


# Source coordinates
source_coordinates_dict = {1: [ [11, 49, 35.66], [22, 23, 48.00] ], 2: [ [4, 16, 9.256], [-24, 4, 11.78] ], 
                     3: [ [4, 16, 9.360], [-24, 4, 12.87] ], 4: [ [4, 16, 8.7084], [-24, 4, 02.945] ],
                     5: [ [1, 37, 23.232], [-8, 27, 52.20] ], 6: [ [15, 50, 00.66], [-78, 11, 09.96] ],
                     7: [ [0, 14, 21.326], [-30, 23, 41.46] ], 8: [ [6, 48, 00.3732], [70, 14, 57.948] ],
                     9: [ [6, 48, 00.3320], [70, 14, 57.761] ], 10: [ [1, 2, 59.488], [-49, 16, 27.910] ],
                     11: [ [2, 39, 52.1308], [-1, 35, 05.755] ], 12: [ [4, 16, 8.84], [-24, 3, 58.62]]}


# Distance from images to MaCL from observations
theta_dict = {1: +0.13, 2: 'nan', 3: 'nan', 4: +0.06, 5: -0.009, 6: +0.55, 7: -0.15, 8: -0.1, 9: -0.32, 10: +7.5e-3, 11: -0.6, 12: -0.05}

# k_star from intracluster light stars. Data obtained from the original papers
k_star_dict = {1: kappa_star_comput(19, lens_redshift[1], source_redshift[1])[0], 
               2: kappa_star_comput(7.943, lens_redshift[2], source_redshift[2])[0], 
               3: kappa_star_comput(7.943, lens_redshift[3], source_redshift[3])[0], 
               4: 0.02, 
               5: kappa_star_comput(10, lens_redshift[5], source_redshift[5])[0], 
               6: 'nan', 
               7: kappa_star_comput(11.55, lens_redshift[7], source_redshift[7])[0], 
               8: 'nan', 
               9: 'nan', 
               10: 'nan',
               11: kappa_star_comput(4.37, lens_redshift[11], source_redshift[11])[0], 
               12: 'nan'}



# Positions of two known symmetric points for determining direction of image elongation in each cluster
# LS2 and LS3 from Bergamini 2020 (13a, 13b)
# LS4 from Caminha 2017 (12.2b, 12.2c)
# LS7 from Mahler 2017 (37.1, 37.2)
# LS8 and LS9 from Chan 2016 (2a, 2b)
# LS10 from Diego 2023 (23.4a, 23.4b)
# LS11 from Lagatutta 2017 (21.1, 21.2)
# LS12 from Diego 2023 (b, b')
reference_positions_dict = {1: [[[11, 49, 35.687], [22, 23, 48.32]], [[11, 49, 35.636], [22, 23, 48.02]]], 
2: [[degrees_to_RA(64.039245), degrees_to_DEC(-24.070383)], [degrees_to_RA(64.038301), degrees_to_DEC(-24.069728)]],
3: [[degrees_to_RA(64.039245), degrees_to_DEC(-24.070383)], [degrees_to_RA(64.038301), degrees_to_DEC(-24.069728)]],
4: [[degrees_to_RA(64.036843), degrees_to_DEC(-24.067457)], [degrees_to_RA(64.036507), degrees_to_DEC(-24.067028)]],
5: [[[1, 37, 23.177], [-8, 27, 51.23]], [[1, 37, 23.292], [-8, 27, 52.96]]],
6: 0,
7: [[degrees_to_RA(3.5890417), degrees_to_DEC(-30.394913)], [degrees_to_RA(3.5887083), degrees_to_DEC(-30.394852)]],
8: [[[6, 48, 0.33], [70, 15, 0.7]], [[6, 48, 0.33], [70, 14, 55.4]]],
9: [[[6, 48, 0.33], [70, 15, 0.7]], [[6, 48, 0.33], [70, 14, 55.4]]],
10: [[degrees_to_RA(15.747970), degrees_to_DEC(-49.274345)], [degrees_to_RA(15.747764), degrees_to_DEC(-49.274494)]],
11: [[degrees_to_RA(39.966733), degrees_to_DEC(-1.5846943)], [degrees_to_RA(39.967252), degrees_to_DEC(-1.5849694)]],
12: [[[4, 16, 8.823], [-24, 3, 58.73]], [[4, 16, 8.773], [-24, 3, 58.05]] ]
}

# $\kappa_0$ and $g^{-1}$ computation

In [15]:
# Star Event
print('Select Star Event')
for event in star_events:
    print(event, ':',  star_events[event])
star_id = int(input(''))

if star_id == 10:   # ElGordo case needs a diferent scaling function, due to the format of its Lens Model
    def redshift_rescale(k_tilda, z_L, z_S):    
        '''Rescales lens model value of kappa (or any other magnification parameter) (k_tilda) for a certain lens redshift (z_L) and source redshift (z_S).
            Returns rescaled value of kappa. ONLY VALID FOR THE ELGORDO CASE, WHERE SKY MAPS ARE GIVEN AT Z=20'''

        # descales from kappa_tilda at z_S=20
        k_descaled = redshift_descale(k_tilda, z_L, 20)

        D_LS = D_A(z_L, z_S)
        D_S = D_A(0, z_S)

        # rescales from kappa_descaled to the corresponding z_S 
        kappa = k_descaled * D_LS[0]  / D_S[0]
        return kappa
else:
    def redshift_rescale(k_tilda, z_L, z_S):    
        '''Rescales lens model value of kappa (or any other magnification parameter) (k_tilda) for a certain lens redshift (z_L) and source redshift (z_S).
            Returns rescaled value of kappa'''

        D_LS = D_A(z_L, z_S)
        D_S = D_A(0, z_S)

        kappa = k_tilda * D_LS[0]  / D_S[0]
        return kappa


# Event information (cluster name, telescope, program, redshifts and source coordinates) from dictionaries above
cluster = clusters[star_id]
telescope = telescopes[star_id]
program = programs[star_id]
z_L = lens_redshift[star_id]
z_S = source_redshift[star_id]                
source_coords = source_coordinates_dict[star_id]


print('\nStar LS{} at coordinates: {}h {}m {:.3f}s {}º {}m {:.3f}s. Lens ({}) redshift: {}. Source redshift: {}'.
      format(star_id, source_coords[0][0], source_coords[0][1], source_coords[0][2],
             source_coords[1][0], source_coords[1][1], source_coords[1][2], cluster, z_L, z_S))

# Arrays for results
kappa_values = np.array([])
kappa_errors = np.array([])
mu_values = np.array([])
mu_errors = np.array([])
d_values = np.array([])
d_errors = np.array([])

## Model opening. This part of the code is for the studied model in particular
# MODEL CHOOSING
try:
    model_ids = model_ids_dict[star_id]
    version_ids = version_ids_dict[star_id]
except:
    print('\n\n\nNo lens model found for the selected event.')
    raise(KeyboardInterrupt)

# If you want to manually choose models for the selected star event, use the code below
# model_ids = []
# version_ids = []
# end_model_selec = False

# print('\nSelect Models:\n')
# while end_model_selec == False:
#     print('\nSelect model:')
#     for model in models:
#         print(model, ':', models[model])
#     print('0: end selection')
#     model_id = int(input(''))

#     if model_id == 0:   # exit model choosing
#         end_model_selec = True
#         break

#     print('\nSelect Version: ', *list(versions.keys()))
#     version_id = float(input(''))    
#     print('')
    
#     try:
#         model = models[model_id]
#         version = versions[version_id]
#         kappa_file   = fits.open('Lens models/{}/{}/{}_{}_model_{}_{}{}_kappa.fits'
#                         .format(cluster, model, telescope, program, cluster, model, version))
#     except:
#         print('No such version for the chosen model found. Lens models/{}/{}/{}_{}_model_{}_{}{}_kappa.fits'
#                         .format(cluster, model, telescope, program, cluster, model, version))
#         continue

#     if model_id in model_ids:
#         idx = model_ids.index(model_id)
#         if version_ids[idx] == version_id:
#             print('Model already chosen!')
#             continue
        
        
#     if model_id != 0:
#         model_ids.append(model_id)
#         version_ids.append(version_id)

        
# Array with the used model ids that have shear information  
shear_model_ids = model_ids.copy()
shear_version_ids = version_ids.copy()


# κ0 AND d COMPUTATION FOR EACH ONE OF THE CHOSEN MODELS
for model_id, version_id in zip(model_ids, version_ids):      
    model = models[model_id]
    version = versions[version_id]
    print('\n ----- Selected Model:  {}{} -----  \n'.format(model, version))

    kappa_file   = fits.open('Lens models/{}/{}/{}_{}_model_{}_{}{}_kappa.fits'
                        .format(cluster, model, telescope, program, cluster, model, version))
    gamma_file   = fits.open('Lens models/{}/{}/{}_{}_model_{}_{}{}_gamma.fits'
                        .format(cluster, model, telescope, program, cluster, model, version))

                         
    shear_info = True                         
    try:
        lambda_file   = fits.open('Lens models/{}/{}/{}_{}_model_{}_{}{}_gamma1.fits'
                                .format(cluster, model, telescope, program, cluster, model, version))
        eta_file   = fits.open('Lens models/{}/{}/{}_{}_model_{}_{}{}_gamma2.fits'
                                .format(cluster, model, telescope, program, cluster, model, version))
    except:
        shear_info = False    

    # Header information
    header_info = kappa_file[0].header
    c_x, c_y = header_info['CRPIX1'], header_info['CRPIX2']
    image_center_deg = [header_info['CRVAL1'], header_info['CRVAL2']]
    delta_x, delta_y = header_info['CDELT1'], header_info['CDELT2']     # degrees
    pixel_size_arcsec = (abs(delta_x) + abs(delta_y))/2 * 3600    # arcsec
    
    delta_RA = delta_x / cos(radians(image_center_deg[1]))
    delta_DEC = delta_y

    # Source positioning
    x_s, y_s = position_to_pixel(source_coords, image_center_deg, c_x, c_y, delta_RA, delta_DEC)
    
    # Magnification parameters calculation at source position
    kappa_map  = kappa_file[0].data
    gamma_map  = gamma_file[0].data
    if shear_info == True:
        lambda_map = lambda_file[0].data
        eta_map    = eta_file[0].data
    else:
        lambda_map = 0
        eta_map = 0

    maps = [kappa_map, gamma_map, lambda_map, eta_map]

    kappa_v, gamma_v, lambda_v, eta_v = magnification_parameters(kappa_map, gamma_map, lambda_map, eta_map,
                                z_L, z_S, source_coords, image_center_deg, c_x, c_y, delta_RA, delta_DEC, shear_info)

    print("At the source position ({}, {}), the obtained magnification parameters are: ".format(x_s+1, y_s+1))
    print("kappa = {:.3f} ± {:.3f}".format(kappa_v[0],kappa_v[1]))
    print("gamma = {:.3f} ± {:.3f}".format(gamma_v[0], gamma_v[1]))
    print("lambda = {:.3f} ± {:.3f}".format(lambda_v[0], lambda_v[1]))
    print("eta = {:.3f} ± {:.3f}".format(eta_v[0], eta_v[1]))
    print('pixel_size = {:.3f}"'.format(delta_DEC*3600))

    magn, magn_err = magnification_at_position(source_coords, z_L, z_S, image_center_deg, c_x, c_y, delta_RA, delta_DEC,
                  kappa_map, gamma_map, lambda_map, eta_map, shear_info)

    print('Magnification at the source position: μ = {:.3f} ± {:.3f}'.format(magn, magn_err))
    mu_values = np.append(mu_values, abs(magn))
    mu_errors = np.append(mu_errors, magn_err)

    ################################### SLOPE COMPUTATION  ##################################################
    
    ref_1_x, ref_1_y = position_to_pixel(reference_positions_dict[star_id][0], image_center_deg, c_x, c_y, delta_RA, delta_DEC)
    ref_2_x, ref_2_y = position_to_pixel(reference_positions_dict[star_id][1], image_center_deg, c_x, c_y, delta_RA, delta_DEC)
    
    if ref_2_x - ref_1_x == 0:
        slope = 9999
    else:
        slope = (ref_2_y - ref_1_y)/(ref_2_x - ref_1_x)
      
    # Critical curve position determination
    theta_max_0 = 1  # arcsec
    delta_theta_0 = 0.1  # arcsec
    tolerance = min(delta_x, delta_y)/10 * 3600 # arcsec

    
    final_crit_positions, final_crit_distance = critical_curve(maps, source_coords, theta_max_0, delta_theta_0, tolerance, 
                            slope, z_L, z_S, image_center_deg, c_x, c_y, delta_RA, delta_DEC, shear_info)

    if ('Na' in final_crit_positions) or (len(final_crit_positions) == 0):
        print('Unavailable to compute critical curve position.')
        idx_remove = shear_model_ids.index(model_id)
        shear_model_ids.remove(model_id)
        shear_version_ids.remove(shear_version_ids[idx_remove])
        continue

    # Conversion to pixels   
    x_lower, y_lower = position_to_pixel(final_crit_positions[0], image_center_deg, c_x, c_y, delta_RA, delta_DEC)
    x_central, y_central = position_to_pixel(final_crit_positions[1], image_center_deg, c_x, c_y, delta_RA, delta_DEC)
    x_upper, y_upper = position_to_pixel(final_crit_positions[2], image_center_deg, c_x, c_y, delta_RA, delta_DEC)

    print('\nMaCL (macro-critical line) coordinates: {}h {}m {:.3f}s {}º {}m {:.3f}s'.format(final_crit_positions[1][0][0], 
                                                                                               final_crit_positions[1][0][1], 
                                                                                               final_crit_positions[1][0][2],
                                                                                               final_crit_positions[1][1][0], 
                                                                                               final_crit_positions[1][1][1], 
                                                                                               final_crit_positions[1][1][2]))
    print('MaCL (macro-critical line) position in pixels: ({}, {})'.format(x_central, y_central))


    kappa_cc, gamma_cc, lambda_cc, eta_cc = magnification_parameters(kappa_map, gamma_map, lambda_map, eta_map,
                                z_L, z_S, final_crit_positions[1], image_center_deg, c_x, c_y, delta_RA, delta_DEC, shear_info)
    
    eig_min = 1 - kappa_cc[0] - gamma_cc[0]
    eig_plus = 1 - kappa_cc[0] + gamma_cc[0]
    
    print('Eigenvalues at CC:  1-κ-γ = {}, 1-κ+γ = {}'.format(round(eig_min, 4), round(eig_plus, 4)))
    
    ################################### g COMPUTATION  ##################################################

    position_0 = final_crit_positions[1]
    d_theta_grads = 10*pixel_size_arcsec # Spacing of 10 pixels
    d_grad, d_grad_err, d_mod = d_grad_comput(maps, position_0, d_theta_grads, slope, z_L, z_S, image_center_deg, 
                                       c_x, c_y, delta_RA, delta_DEC, shear_info)


    print('Magnitude of gradient g = -grad(κ+γ): g_value = {:.3f} ± {:.3f} arcmin^-1'.format(d_mod[0]*60, d_mod[1]*60))
    d_values = np.append(d_values, d_mod[0])
    d_errors = np.append(d_errors, d_mod[1])

    kappa_values = np.append(kappa_values, kappa_v[0])
    kappa_errors = np.append(kappa_errors, kappa_v[1])
    
    print('\n Final values for the studied model:    κ_0 = {:.3f} \t g = {:.3f} arcmin^-1'.format(kappa_v[0], d_mod[0]*60))

    

# FINAL κ0 VALUE AT SOURCE POSITION COMPUTATION  
kappa_final = np.mean(kappa_values)

k_err_sist_2 = 0
for k_err in kappa_errors:
    k_err_sist_2 += k_err**2
k_err_sist = k_err_sist_2**0.5

k_err_rand = np.std(kappa_values)/(len(kappa_values))**0.5
kappa_final_err = (k_err_sist**2 + k_err_rand**2)**0.5 


# FINAL μ VALUE AT SOURCE POSITION COMPUTATION  
mu_final = np.mean(mu_values)
mu_err_sist_2 = 0
for mu_err in mu_errors:
    mu_err_sist_2 += mu_err**2

mu_err_sist = mu_err_sist_2**0.5
mu_err_rand = np.std(mu_values)/(len(mu_values))**0.5
mu_final_err = (mu_err_sist**2 + mu_err_rand**2)**0.5 

    
print('\n---------------------------------------------------------------------------')

print('\nFinal κ_0 = {:.3f} ± {:.3f} \t\t  using {} models.'.format(kappa_final, kappa_final_err, len(model_ids)))

# FINAL d VALUE AT SOURCE POSITION COMPUTATION  
if len(d_values) != 0:
    d_final = np.mean(d_values)              # average over d, not d^-1     
    d_err_sist_2 = 0
    for d_err in d_errors:
        d_err_sist_2 += d_err**2

    d_err_sist = d_err_sist_2**0.5
    d_err_rand = np.std(d_values)/(len(d_errors))**0.5
    d_final_err = (d_err_sist**2 + d_err_rand**2)**0.5
        
    print('Final g = {:.3f} ± {:.3f} arcmin^(-1)  \t\t  using {} models.'.format(d_final*60, d_final_err*60, len(shear_model_ids)))

    print('And therefore, the final inverse gradient value is:  1/g = {:.3f} arcsec'.format(1/d_final))
     
#####    Use code below for saving obtained results in  Magnification parameters table.dat   ####

try: # try to read existing data file
    mag_table = pd.read_csv('Numerical results/Magnification parameters table.dat', sep = '\t')

except:  # create new file if no file is found
    mag_values = {'i': [], '  Star Event / Lensing Cluster / Original Paper  ': [], 'z_L': [], 
                  'z_S': [], '    κ_0    ': [], '    δκ_0    ': [], 
                    '    d (arcmin^-1)    ': [], '    δd (arcmin^-1)    ': [], '    |μ|    ': [], '    δ|μ|    ': []}
    mag_table = pd.DataFrame.from_dict(mag_values)

try:  # dropping of unwanted columns
    mag_table = mag_table.drop('Unnamed: 0', axis = 1)
except:
    pass
try: # dropping of unwanted columns
    mag_table = mag_table.drop('Unnamed: 0.1', axis = 1)
except:
    pass


update = int(input('\nDo you want to update the current magnification parameter list? Yes [1] / No [0]'))

if update == 1:
    star_ids_table = np.array(mag_table['i'])
    if star_id in star_ids_table:
        idx = int(np.where(star_ids_table == star_id)[0])
        mag_table = mag_table.drop(idx, axis = 0)
    mag_values = {'i': star_id, '  Star Event / Lensing Cluster / Original Paper  ': star_events[star_id], 
                  'z_L': z_L, 'z_S': z_S, '    κ_0    ': kappa_final, '    δκ_0    ': kappa_final_err, 
                    '    d (arcmin^-1)    ': d_final*60, '    δd (arcmin^-1)    ': d_final_err*60,
                 '    |μ|    ': mu_final, '    δ|μ|    ': mu_final_err}
    mag_table = mag_table.append(mag_values, ignore_index = True)
    mag_table = mag_table.sort_values(by=['i'])
    
    mag_table.to_csv('Numerical results/Magnification parameters table.dat', sep = '\t')


Select Star Event
1 : LS1 (Icarus) lensed by MACSJ1149 (Kelly 2018)
2 : LS2 (NW Spock) lensed by MACSJ0416 (Rodney 2018)
3 : LS3 (SE Spock) lensed by MACSJ0416 (Rodney 2018)
4 : LS4 (Warhol) lensed by MACSJ0416 (Kaurov 2019 / Chen 2019)
5 : LS5 (Earendel) lensed by WHL0137 (Welch 2022)
6 : LS6 (Godzilla) lensed by PSZ1 G311.65 (Diego 2022)
7 : LS7 (...) lensed by Abell 2744 (Chen 2022)
8 : LS8 (star-1) lensed by MACSJ0647 (Meena 2023)
9 : LS9 (star-2) lensed by MACSJ0647 (Meena 2023)
10 : LS10 (Quyllur) lensed by El Gordo (Diego 2023)
11 : LS11 (...) lensed by Abell 370 (Meena 2023)
12 : LS12 (Mothra) lensed by MACSJ0416 (Diego 2023)
12

Star LS12 at coordinates: 4h 16m 8.840s -24º 3m 58.620s. Lens (macs0416) redshift: 0.397. Source redshift: 2.091

 ----- Selected Model:  zitrin-ltm-gauss_v3 -----  

At the source position (1032, 1657), the obtained magnification parameters are: 
kappa = 0.936 ± 0.005
gamma = 0.100 ± 0.002
lambda = -0.010 ± 0.005
eta = 0.099 ± 0.001
pixel_size = 0.060

C:\Users\Claudi\AppData\Local\Temp\ipykernel_16460\1363877711.py:315: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mag_table = mag_table.append(mag_values, ignore_index = True)
